In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)

import seaborn as sns
sns.set_theme(style = "darkgrid")

import matplotlib.pyplot as plt

In [ ]:
telco = pd.read_csv("../data/Telco-Customer-Churn.csv")

In [ ]:
telco.head()

In [ ]:
rows = telco.shape[0]
columns = telco.shape[1]

print(f"rows: {rows}")
print(f"columns: {columns}\n")
print(f"total duplicates {telco.duplicated().sum()}\n")

for column in telco.columns:
    print(f"#### {column} ####")
    print(f"data type: {telco[column].dtype}")
    print(f"number of unique values: {telco[column].nunique()}")
    if column not in ["TotalCharges", "MonthlyCharges", "customerID", "tenure"]:
        print(f"unique values: {telco[column].unique()}")
    print(f"missing values {telco[column].isna().sum()}\n")

All columns appear to be clean apart from `TotalCharges`, which is labelled as `str` rather than `float`. As the five viewed values are `float`, it is likely that some entries elsewhere are non-numeric, causing pandas to store the column as a `str`. 

In [ ]:
non_float = pd.to_numeric(
    telco["TotalCharges"], 
    errors = "coerce"
).isna()

print(len(telco[non_float]))
telco[non_float]

There are a total of $11$ rows which contain `" "` values. At first glance, there does not appear to be a consistent pattern across these records, although $8/11$ rows use `PaymentMethod` `Mailed check`, and all of them have $0$ `tenure`. For now, these values will be left as `null`. During the EDA, the distributions and relationships between variables will be examined, which should provide insight into the most appropriate method for imputing the missing values. 

In [ ]:
telco.loc[non_float, "TotalCharges"] = pd.NA
telco["TotalCharges"] = telco["TotalCharges"].astype(float)

## EDA

We start of by mapping the `SeniorCitizen` values to `Yes` and `No` as this will be treated as a categorical column. We remove `customerID` since it is not useful for modelling, and finally we split the data between categorical and numerical data types. 

In [ ]:
senior_citizen_map = {
    0: "No", 
    1: "Yes"
}

telco["SeniorCitizen"] = telco["SeniorCitizen"].map(senior_citizen_map)
telco = telco.drop(columns = ["customerID"])

categorical = telco.select_dtypes("str")
numerical = telco.select_dtypes(["float64", "int64"])
numerical["Churn"] = telco["Churn"]

### Numerical

In [ ]:
columns = numerical.drop(columns = ["Churn"]).columns.tolist()

fig, axes = plt.subplots(
    nrows = 1, 
    ncols = 3, 
    figsize = (12, 4), 
    constrained_layout = True
)

axes = axes.flatten()
fig.suptitle("Distribution of Numeric Features by Churn Status")

for ax, column in zip(axes, columns):
    sns.histplot(
        data = numerical, 
        x = column, 
        hue = "Churn",
        ax = ax
    )

plt.show()

In [ ]:
columns = numerical.drop(columns = ["Churn"]).columns.tolist()

fig, axes = plt.subplots(
    nrows = 1, 
    ncols = 3, 
    figsize = (12, 4), 
    constrained_layout = True
)

axes = axes.flatten()
fig.suptitle("Comparison of Numeric Features Across Churn Groups")

for ax, column in zip(axes, columns):
    sns.boxplot(
        data = numerical, 
        x = column, 
        y = "Churn",
        ax = ax
    )

plt.show()

In [ ]:
graph = sns.pairplot(
    data = numerical,
    hue = "Churn",
    kind = "kde",
    height = 2.5,
    aspect = 1
)

graph.fig.suptitle(
    "Numerical Feature Distributions and Relationships by Churn Status",
    y = 1.01)

plt.show()

In [ ]:
churn_map = {
    "No": 0, 
    "Yes": 1,
    0: 0,
    1: 1
}

numerical["Churn"] = numerical["Churn"].map(churn_map)

corr = numerical.corr()

plt.figure(
    figsize = (6, 4), 
    constrained_layout = True
)

plt.title("Correlation Between Numerical Features")

sns.heatmap(
    data = corr,
    annot = True,
    cmap = "crest"
)

plt.show()

### Categorical

In [ ]:
plt.figure(
    figsize = (6, 4), 
    constrained_layout = True
)

plt.title("Customer Distribution by Churn Status")

sns.countplot(
    data = categorical, 
    x = "Churn",
    stat = "percent"
)

plt.show()

In [ ]:
columns = categorical.drop(columns = ["Churn"]).columns.tolist()

fig, axes = plt.subplots(
    nrows = 4, 
    ncols = 4, 
    figsize = (16, 12), 
    constrained_layout = True
)

axes = axes.flatten()
fig.suptitle("Comparison of Numeric Features Across Churn Groups")

for ax, column in zip(axes, columns):
    sns.countplot(
        data = categorical,
        x = column, 
        ax = ax
    )
    ax.tick_params(axis = "x", rotation = 12.5)

plt.show()

In [ ]:
columns = categorical.drop(columns = ["Churn"]).columns.tolist()

fig, axes = plt.subplots(
    nrows = 4, 
    ncols = 4, 
    figsize = (16, 12), 
    constrained_layout = True
)

axes = axes.flatten()
fig.suptitle("Comparison of Numeric Features Across Churn Groups")

for ax, column in zip(axes, columns):
    sns.countplot(
        data = categorical,
        x = column, 
        hue = "Churn",
        stat = "count",
        ax = ax
    )
    ax.tick_params(axis = "x", rotation = 12.5)

plt.show()

In [ ]:
columns = categorical.drop(columns = ["Churn"]).columns.tolist()

fig, axes = plt.subplots(
    nrows = 4, 
    ncols = 4, 
    figsize = (16, 12), 
    constrained_layout = True
)

axes = axes.flatten()
fig.suptitle("Churn Rate Across Numeric Features")

for ax, column in zip(axes, columns):
    churn_rate = categorical.groupby([column, "Churn"]).size().unstack(level = -1).reset_index()
    churn_rate["Churn Rate (%)"] = 100.0 * churn_rate["Yes"] / (churn_rate["Yes"] + churn_rate["No"])

    sns.barplot(
        data = churn_rate,
        x = column,
        y = "Churn Rate (%)",
        ax = ax
    )

